# iter5a2 — RECIPE arm on Scale A's corpus: the small line's own winning recipe at 6M scale

**Same Drive data as iter5a — NOTHING new to upload.**
Base `small128_vh2` (20k bar: 13,334 / 9,364 / 3.9%). Corpus `iter5.pt` (6,175,263 states).
**Recipe = gate-3/vh1 winner, never tested above 394k states:** blend 0.5 hard-CE, T=1.0,
dw=0, lr 1e-4, 3 epochs, step saves every 500.
Reference: iter5a (dw3/T0.7/soft/lr3e-4) read -32% at matched steps (HISTORY 199).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v4.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v4.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/iter5.pt.gz /content/iter5.pt.gz
gz=os.path.getsize('/content/iter5.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 407_137_388, f'.gz truncated! got {gz}; re-upload iter5.pt.gz'
!gunzip -t /content/iter5.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/iter5.pt.gz > /content/alphatrain/data/iter5.pt
pt=os.path.getsize('/content/alphatrain/data/iter5.pt')
assert pt == 2_365_130_759, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 6,175,263 states ({time.time()-t0:.0f}s)')
!rm /content/iter5.pt.gz
!cp {DRIVE}/small128_vh2.pt /content/alphatrain/data/
assert os.path.getsize('/content/alphatrain/data/small128_vh2.pt') == 12_067_557
!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG (gate-3 winning recipe at 6M scale) =====
CHANNELS = 128
EPOCHS   = 3
BATCH    = 8192
LR       = 1e-4
T        = 1.0
DW       = 0
BLEND    = 0.5
SEED     = 42
SAVE_STEPS = 500
RUN      = "iter5a2"
print(f'RUN={RUN} epochs={EPOCHS} batch={BATCH} lr={LR} T={T} dw={DW} blend={BLEND}')


In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/iter5.pt \
    --resume alphatrain/data/small128_vh2.pt --warm-start \
    --channels {CHANNELS} --seed {SEED} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DW} --blend-alpha {BLEND} \
    --save-every-steps {SAVE_STEPS} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/checkpoints/{RUN} 2>&1 | tee /content/{RUN}_train.log
# val is informational only — the gate is GAMEPLAY (floor-first, 5k decides).

In [ ]:
import shutil, os, glob
DRIVE='/content/drive/MyDrive/alphatrain'
for pat in ['epoch_*.pt', 'e*_s*.pt']:
    for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/{pat}')):
        dst=f'{DRIVE}/{RUN}_{os.path.basename(f)}'; shutil.copy(f,dst); print('Saved', dst)
for f in ['best.pt','latest.pt']:
    s=f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(s): shutil.copy(s,f'{DRIVE}/{RUN}_{f}'); print('Saved', f'{DRIVE}/{RUN}_{f}')

## Gate protocol (M5, C++ eval — download checkpoints as they save)

```bash
python -m alphatrain.inference_cpp.export_ts --model alphatrain/data/iter5a_e1_s400.pt
mv alphatrain/inference_cpp/data/policy_ts.pt alphatrain/inference_cpp/data/dagger1_e1_s400_ts.pt
cd alphatrain/inference_cpp
./build/eval --model data/dagger1_e1_s400_ts.pt --device mps --seed-start 775000 --seed-end 775500 --batch 500
```
1. **Screen step checkpoints** (s100..s900 + epoch saves) at 500 seeds. The 500-seed screen is a
   **catastrophe filter ONLY** — it rejects blowups; it does NOT rank close calls (vh1 itself read
   P50 −6% at 500 seeds, then WON +4.6% at 5k — HISTORY:3375).
2. Take the 2-3 best-looking survivors (floor-first: <1000%, P5/P10) to the **5k eval**
   (`--seed-end 780000`) against vh1's bar: mean 13,080 / P50 9,323 / P5 1,222 / <1000 3.5%.
3. Expected per the MICRO-GO calibration: +3-8%% median. If it clears → name it `small128_vh2`,
   retrain the survival head on ITS backbone, re-export, then RE-GATE Engine B (own-MCTS judge)
   at the new level — the alternation plan (memory: project_small_model_distill).
4. All step checkpoints regress at 500 (catastrophically) → recheck mix %% and label sanity
   before ANY recipe surgery (HISTORY 178 lesson: measure, don't theorize).